# Colab LTX-2.3 T2V/I2V + ngrok API

啟動 ComfyUI，載入 LTX-2.3 Text-to-Video / Image-to-Video 工作流，並用 ngrok 暴露 Web UI 與 ComfyUI API。

流程：
1. 安裝 ComfyUI、custom nodes 與必要套件。
2. 下載 LTX-2.3 模型與 workflow JSON。
3. 啟動 ngrok tunnel。
4. 啟動 ComfyUI。
5. 從外部程式呼叫 `/prompt`、`/history/{prompt_id}`、`/view` 產生並下載影片。


In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

### 1. Install ComfyUI + dependencies + custom nodes

In [ ]:
# Install ComfyUI + dependencies + custom nodes
%cd /content
![ -d ComfyUI ] || git clone https://github.com/comfyanonymous/ComfyUI.git

%cd /content/ComfyUI
!pip -q install -r requirements.txt pyngrok xformers
!pip -q install torchsde einops diffusers accelerate av spandrel albumentations onnx opencv-python onnxruntime tqdm ipywidgets
!apt-get -y install -qq aria2 > /dev/null 2>&1

%cd /content/ComfyUI/custom_nodes
![ -d ComfyUI-KJNodes ] || git clone https://github.com/kijai/ComfyUI-KJNodes
![ -d ComfyUI-GGUF ] || git clone https://github.com/city96/ComfyUI-GGUF
![ -d ComfyUI-LTXVideo ] || git clone https://github.com/Lightricks/ComfyUI-LTXVideo
![ -d ComfyUI-VideoHelperSuite ] || git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite
![ -d ComfyUI-MelBandRoFormer ] || git clone https://github.com/kijai/ComfyUI-MelBandRoFormer

# Install custom-node requirements.
import subprocess
from pathlib import Path

custom_nodes = [
    "ComfyUI-KJNodes",
    "ComfyUI-GGUF",
    "ComfyUI-LTXVideo",
    "ComfyUI-VideoHelperSuite",
    "ComfyUI-MelBandRoFormer",
]

for node in custom_nodes:
    req = Path("/content/ComfyUI/custom_nodes") / node / "requirements.txt"
    if req.exists():
        print(f"Installing requirements for {node}")
        subprocess.run(["python", "-m", "pip", "install", "-q", "-r", str(req)], check=False)
    else:
        print(f"No requirements.txt for {node}")


# Install specific version of Kornia for ComfyUI-LTXVideo
subprocess.run([
    "python", "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps", "kornia==0.8.2"
], check=True)

# Sanity check before launching ComfyUI.
import kornia
from kornia.geometry.transform.pyramid import (
    PyrUp,
    build_laplacian_pyramid,
    build_pyramid,
    find_next_powerof_two,
    is_powerof_two,
    pad,
)
print("Kornia version:", kornia.__version__)
print("Kornia pyramid import check: OK")

### 2. Download LTX-2.3 models + workflow

In [ ]:
%cd /content/ComfyUI

# Download LTX-2.3 models + workflow
!mkdir -p models/unet models/text_encoders models/vae models/latent_upscale_models models/diffusion_models models/loras

# Main LTX-2.3 model
!test -f models/unet/ltx-2.3-22b-dev-Q4_K_M.gguf || aria2c --console-log-level=error -c -x 16 -s 16 -k 1M -d models/unet -o ltx-2.3-22b-dev-Q4_K_M.gguf https://huggingface.co/unsloth/LTX-2.3-GGUF/resolve/main/ltx-2.3-22b-dev-Q4_K_M.gguf

# Text encoder + connector
!test -f models/text_encoders/gemma-3-12b-it-q4_0_s.gguf || aria2c --console-log-level=error -c -x 16 -s 16 -k 1M -d models/text_encoders -o gemma-3-12b-it-q4_0_s.gguf https://huggingface.co/Dampfinchen/google-gemma-3-12b-it-qat-q4_0-gguf-small-fix/resolve/main/gemma-3-12b-it-q4_0_s.gguf
!test -f models/text_encoders/ltx-2.3-22b-dev_embeddings_connectors.safetensors || aria2c --console-log-level=error -c -x 16 -s 16 -k 1M -d models/text_encoders -o ltx-2.3-22b-dev_embeddings_connectors.safetensors https://huggingface.co/unsloth/LTX-2.3-GGUF/resolve/main/text_encoders/ltx-2.3-22b-dev_embeddings_connectors.safetensors

# VAE / upscaler / helper models
!test -f models/vae/ltx-2.3-22b-dev_video_vae.safetensors || aria2c --console-log-level=error -c -x 16 -s 16 -k 1M -d models/vae -o ltx-2.3-22b-dev_video_vae.safetensors https://huggingface.co/unsloth/LTX-2.3-GGUF/resolve/main/vae/ltx-2.3-22b-dev_video_vae.safetensors
!test -f models/vae/ltx-2.3-22b-dev_audio_vae.safetensors || aria2c --console-log-level=error -c -x 16 -s 16 -k 1M -d models/vae -o ltx-2.3-22b-dev_audio_vae.safetensors https://huggingface.co/unsloth/LTX-2.3-GGUF/resolve/main/vae/ltx-2.3-22b-dev_audio_vae.safetensors
!test -f models/vae/taeltx2_3.safetensors || aria2c --console-log-level=error -c -x 16 -s 16 -k 1M -d models/vae -o taeltx2_3.safetensors https://huggingface.co/Kijai/LTX2.3_comfy/resolve/main/vae/taeltx2_3.safetensors
!test -f models/latent_upscale_models/ltx-2.3-spatial-upscaler-x2-1.0.safetensors || aria2c --console-log-level=error -c -x 16 -s 16 -k 1M -d models/latent_upscale_models -o ltx-2.3-spatial-upscaler-x2-1.0.safetensors https://huggingface.co/Lightricks/LTX-2.3/resolve/main/ltx-2.3-spatial-upscaler-x2-1.0.safetensors
!test -f models/diffusion_models/MelBandRoformer_fp16.safetensors || aria2c --console-log-level=error -c -x 16 -s 16 -k 1M -d models/diffusion_models -o MelBandRoformer_fp16.safetensors https://huggingface.co/Kijai/MelBandRoFormer_comfy/resolve/main/MelBandRoformer_fp16.safetensors
!test -f models/loras/ltx-2.3-22b-distilled-lora-384.safetensors || aria2c --console-log-level=error -c -x 16 -s 16 -k 1M -d models/loras -o ltx-2.3-22b-distilled-lora-384.safetensors https://huggingface.co/Lightricks/LTX-2.3/resolve/main/ltx-2.3-22b-distilled-lora-384.safetensors

# API-format workflow used by the external API example below
!test -f ltx2_3_t2v_i2v.json || wget -q --show-progress -O ltx2_3_t2v_i2v.json "https://raw.githubusercontent.com/AICHUCKY/Comfyui-Workflows/AICHUCKY-patch-1/Ltx2.3%20.json"

### 3. Start ngrok tunnel

In [ ]:
from getpass import getpass
from pyngrok import ngrok

try:
    from google.colab import userdata
    token = userdata.get("NGROK_AUTHTOKEN")
except Exception:
    token = None

if not token:
    token = getpass("NGROK_AUTHTOKEN: ")

ngrok.set_auth_token(token)
ngrok.kill()

public_url = ngrok.connect(8188, "http").public_url
print("ComfyUI Web UI:", public_url)
print("ComfyUI API:", public_url + "/prompt")
print("History API:", public_url + "/history/{prompt_id}")


### 4. Launch ComfyUI

In [ ]:
%cd /content/ComfyUI

# If you have enough VRAM, you may remove --cache-none.
!python main.py \
  --listen 0.0.0.0 \
  --port 8188 \
  --lowvram \
  --cache-none \
  --enable-cors-header '*'

## 外部呼叫 API 範例

- 先執行前面的 cell，取得 ngrok URL。
- 把 `BASE` 改成你的 ngrok URL。
- `MODE = "t2v"` 代表 Text-to-Video。
- `MODE = "i2v"` 代表 Image-to-Video，並設定 `INPUT_IMAGE`。
- 這個範例會提交 workflow、輪詢 `/history/{prompt_id}`，最後用 `/view` 下載輸出影片。


In [ ]:
import io
import json
import time
import uuid
import requests
from pathlib import Path
from PIL import Image

BASE = "https://你的-ngrok-url.ngrok-free.app"
HEADERS = {"ngrok-skip-browser-warning": "1"}

MODE = "t2v"  # "t2v" or "i2v"
INPUT_IMAGE = "input.png"  # 適用於 MODE = "i2v"

PROMPT = '''\
A cinematic romantic scene of a beautiful young adult Chinese college student and her boyfriend on a wide street lined with cherry blossom trees. Cherry blossom petals are continuously falling. The young woman happily makes small playful hops beside her boyfriend, then moves in front of him, turns back, and gives him a sweet beautiful smile. After that, she slowly runs away in a playful romantic way, making it feel like her boyfriend is chasing her. Dreamy atmosphere, soft spring daylight, gentle breeze, flowing hair, realistic motion, smooth camera tracking, shallow depth of field, poetic youthful love mood, high quality.'''

WIDTH = 480
HEIGHT = 832
SECONDS = 5 # 可以先從 5 秒開始
FPS = 24

SEED_PASS_1 = 43
SEED_PASS_2 = 42
CFG_PASS_1 = 1.0
CFG_PASS_2 = 1.0

# 要從安裝套件那邊的 JSON 檔案連結下載
WORKFLOW_PATH = "ltx2_3_t2v_i2v.json"

# 寬/高至少為 256，並且是 32 的倍數
def safe_size(value):
    return max(256, round(value / 32) * 32)

# 上傳圖片，支援 PIL.Image 或檔案路徑
def upload_image(path_or_image, filename):
    if isinstance(path_or_image, Image.Image):
        buffer = io.BytesIO()
        path_or_image.save(buffer, format="PNG")
        buffer.seek(0)
        files = {"image": (filename, buffer, "image/png")}
    else:
        path = Path(path_or_image)
        files = {"image": (path.name, path.open("rb"), "application/octet-stream")}

    # 上傳圖片到伺服器，並回傳伺服器上儲存的檔案名稱
    response = requests.post(
        f"{BASE}/upload/image",
        files=files,
        data={"overwrite": "true"},
        headers=HEADERS,
        timeout=None
    )
    response.raise_for_status()

    result = response.json()
    return result.get("name", filename)

# 將工作流程送到伺服器，並回傳 prompt_id
def queue_prompt(workflow):
    response = requests.post(
        f"{BASE}/prompt",
        json={"prompt": workflow, "client_id": str(uuid.uuid4())},
        headers=HEADERS,
        timeout=None
    )
    response.raise_for_status()
    return response.json()["prompt_id"]

# 等待伺服器完成工作流程，並回傳歷史紀錄
def wait_for_history(prompt_id, sleep_seconds=10):
    while True:
        try:
            r = requests.get(
                f"{BASE}/history/{prompt_id}",
                headers=HEADERS,
                timeout=(10, 60),
            )
            r.raise_for_status()

            history = r.json()
            if prompt_id in history:
                return history[prompt_id]

            print("still running...")

        except requests.exceptions.RequestException as e:
            print("history request failed, retrying:", type(e).__name__)

        time.sleep(sleep_seconds)

# 下載伺服器上的輸出檔案，並儲存到本地端
def download_outputs(history, output_dir="outputs"):
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)

    saved = []

    for output in history["outputs"].values():
        for key in ("videos", "gifs", "images"):
            for item in output.get(key, []):
                filename = item.get("filename", f"output_{len(saved)}")
                path = output_dir / filename

                while True:
                    try:
                        r = requests.get(
                            f"{BASE}/view",
                            params=item,
                            headers=HEADERS,
                            timeout=(10, 300),
                        )
                        r.raise_for_status()
                        path.write_bytes(r.content)
                        saved.append(path)
                        print("saved:", path)
                        break

                    except requests.exceptions.RequestException as e:
                        print("download failed, retrying:", type(e).__name__)
                        time.sleep(10)

    return saved

# 寬/高至少為 256，並且是 32 的倍數
width = safe_size(WIDTH)
height = safe_size(HEIGHT)

# 讀取工作流程 JSON 檔案
workflow = json.load(open(WORKFLOW_PATH, encoding="utf-8"))

# Common settings
workflow["292"]["inputs"]["value"] = width
workflow["293"]["inputs"]["value"] = height
workflow["285"]["inputs"]["value"] = FPS
workflow["121"]["inputs"]["text"] = PROMPT
workflow["291"]["inputs"]["value"] = SECONDS

# Pass 1
workflow["137"]["inputs"]["sampler_name"] = "lcm"
workflow["360"]["inputs"]["sigmas"] = "1.0, 0.99375, 0.9875, 0.98125, 0.975, 0.909375, 0.725, 0.421875, 0.0"
workflow["129"]["inputs"]["cfg"] = CFG_PASS_1
workflow["115"]["inputs"]["noise_seed"] = SEED_PASS_1

# Pass 2
workflow["138"]["inputs"]["sampler_name"] = "euler_cfg_pp"
workflow["359"]["inputs"]["sigmas"] = "0.85, 0.7250, 0.4219, 0.0"
workflow["103"]["inputs"]["cfg"] = CFG_PASS_2
workflow["114"]["inputs"]["noise_seed"] = SEED_PASS_2

if MODE.lower() == "t2v": # 文字轉影片
    workflow["290"]["inputs"]["value"] = True

    # Some LTX workflows still contain an image-loader node.
    # Uploading a black dummy image prevents missing-input errors.
    dummy = Image.new("RGB", (width, height), color="black")
    workflow["167"]["inputs"]["image"] = upload_image(dummy, "ltx_dummy.png")

elif MODE.lower() == "i2v": # 圖片轉影片
    workflow["290"]["inputs"]["value"] = False
    workflow["167"]["inputs"]["image"] = upload_image(INPUT_IMAGE, Path(INPUT_IMAGE).name)

else: # 拋出錯誤
    raise ValueError('MODE must be "t2v" or "i2v".')

# 送出工作流程，取得 prompt_id
prompt_id = queue_prompt(workflow)
print("prompt_id:", prompt_id)

# 等待伺服器完成工作流程，並下載輸出檔案
history = wait_for_history(prompt_id)
saved_files = download_outputs(history)
